In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

appointments = pd.read_csv(RAW_DIR / "appointments.csv")
patients = pd.read_csv(RAW_DIR / "patients.csv")
doctors = pd.read_csv(RAW_DIR / "doctors.csv")
slots = pd.read_csv(RAW_DIR / "slots.csv")

print("appointments:", appointments.shape)
print("patients:", patients.shape)
print("doctors:", doctors.shape)
print("slots:", slots.shape)

appointments.head()

appointments: (6837, 18)
patients: (5426, 4)
doctors: (24, 4)
slots: (9432, 4)


,appointment_id,slot_id,scheduling_date,appointment_date,appointment_time,scheduling_interval,status,check_in_time,appointment_duration,start_time,end_time,waiting_time,patient_id,sex,age,age_group,department,doctor_id
0,6,1,2023-12-08,2024-01-01,08:00:00,24,attended,07:35:33,46.5,08:01:05,08:47:35,25.5,527,Female,23,20-24,General Medicine,D010
1,118,21,2023-12-25,2024-01-01,13:00:00,7,did not attend,NaN,NaN,NaN,NaN,NaN,320,Female,17,15-19,Neurology,D024
2,194,22,2023-12-29,2024-01-01,13:15:00,3,attended,13:01:40,16.9,13:28:46,13:45:40,27.1,666,Female,67,65-69,Pediatrics,D018
3,212,23,2023-12-30,2024-01-01,13:30:00,2,attended,13:09:37,8.3,13:46:43,13:55:01,37.1,5154,Male,90,90+,Orthopedics,D016
4,168,24,2023-12-28,2024-01-01,13:45:00,4,attended,13:16:48,21.8,13:55:56,14:17:44,39.1,4135,Male,60,60-64,Dermatology,D005


In [2]:
print("Appointments columns:")
print(appointments.columns.tolist())

print("\nPatients columns:")
print(patients.columns.tolist())

print("\nDoctors columns:")
print(doctors.columns.tolist())

print("\nSlots columns:")
print(slots.columns.tolist())

Appointments columns:
['appointment_id', 'slot_id', 'scheduling_date', 'appointment_date', 'appointment_time', 'scheduling_interval', 'status', 'check_in_time', 'appointment_duration', 'start_time', 'end_time', 'waiting_time', 'patient_id', 'sex', 'age', 'age_group', 'department', 'doctor_id']

Patients columns:
['patient_id', 'name', 'sex', 'dob']

Doctors columns:
['doctor_id', 'doctor_name', 'department', 'max_appointments_per_day']

Slots columns:
['slot_id', 'appointment_date', 'appointment_time', 'is_available']


In [3]:
# Make copies
appointments_clean = appointments.copy()
patients_clean = patients.copy()
doctors_clean = doctors.copy()
slots_clean = slots.copy()

# Convert date columns
appointments_clean["scheduling_date"] = pd.to_datetime(appointments_clean["scheduling_date"])
appointments_clean["appointment_date"] = pd.to_datetime(appointments_clean["appointment_date"])
patients_clean["dob"] = pd.to_datetime(patients_clean["dob"])
slots_clean["appointment_date"] = pd.to_datetime(slots_clean["appointment_date"])

# Convert time columns
time_columns = ["appointment_time", "check_in_time", "start_time", "end_time"]

for col in time_columns:
    appointments_clean[col] = pd.to_datetime(
        appointments_clean[col],
        format="%H:%M:%S",
        errors="coerce"
    ).dt.time

# Convert numeric columns
numeric_columns = [
    "scheduling_interval",
    "appointment_duration",
    "waiting_time",
    "age"
]

for col in numeric_columns:
    appointments_clean[col] = pd.to_numeric(appointments_clean[col], errors="coerce")

appointments_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 6837 entries, 0 to 6836
Data columns (total 18 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   appointment_id        6837 non-null   int64         
 1   slot_id               6837 non-null   int64         
 2   scheduling_date       6837 non-null   datetime64[us]
 3   appointment_date      6837 non-null   datetime64[us]
 4   appointment_time      6837 non-null   object        
 5   scheduling_interval   6837 non-null   int64         
 6   status                6837 non-null   str           
 7   check_in_time         5182 non-null   object        
 8   appointment_duration  5182 non-null   float64       
 9   start_time            5182 non-null   object        
 10  end_time              5182 non-null   object        
 11  waiting_time          5182 non-null   float64       
 12  patient_id            6837 non-null   int64         
 13  sex                   6837 no

In [4]:
# Create useful date/time features
appointments_clean["appointment_year"] = appointments_clean["appointment_date"].dt.year
appointments_clean["appointment_month"] = appointments_clean["appointment_date"].dt.month
appointments_clean["appointment_month_name"] = appointments_clean["appointment_date"].dt.month_name()
appointments_clean["appointment_day_name"] = appointments_clean["appointment_date"].dt.day_name()

# Extract hour from appointment_time
appointments_clean["appointment_hour"] = appointments_clean["appointment_time"].apply(
    lambda x: x.hour if pd.notnull(x) else np.nan
)

# Outcome flags
appointments_clean["is_attended"] = appointments_clean["status"].eq("attended").astype(int)
appointments_clean["is_no_show"] = appointments_clean["status"].eq("did not attend").astype(int)
appointments_clean["is_cancelled"] = appointments_clean["status"].eq("cancelled").astype(int)

# Waiting time category
def categorize_waiting_time(wait):
    if pd.isna(wait):
        return "Not attended"
    elif wait <= 15:
        return "0-15 min"
    elif wait <= 30:
        return "16-30 min"
    elif wait <= 45:
        return "31-45 min"
    elif wait <= 60:
        return "46-60 min"
    else:
        return "60+ min"

appointments_clean["waiting_time_category"] = appointments_clean["waiting_time"].apply(categorize_waiting_time)

appointments_clean.head()

,appointment_id,slot_id,scheduling_date,appointment_date,appointment_time,scheduling_interval,status,check_in_time,appointment_duration,start_time,...,doctor_id,appointment_year,appointment_month,appointment_month_name,appointment_day_name,appointment_hour,is_attended,is_no_show,is_cancelled,waiting_time_category
0,6,1,2023-12-08,2024-01-01,08:00:00,24,attended,07:35:33,46.5,08:01:05,...,D010,2024,1,January,Monday,8,1,0,0,16-30 min
1,118,21,2023-12-25,2024-01-01,13:00:00,7,did not attend,NaT,NaN,NaT,...,D024,2024,1,January,Monday,13,0,1,0,Not attended
2,194,22,2023-12-29,2024-01-01,13:15:00,3,attended,13:01:40,16.9,13:28:46,...,D018,2024,1,January,Monday,13,1,0,0,16-30 min
3,212,23,2023-12-30,2024-01-01,13:30:00,2,attended,13:09:37,8.3,13:46:43,...,D016,2024,1,January,Monday,13,1,0,0,31-45 min
4,168,24,2023-12-28,2024-01-01,13:45:00,4,attended,13:16:48,21.8,13:55:56,...,D005,2024,1,January,Monday,13,1,0,0,31-45 min


In [5]:
# Merge doctor details into appointments
appointments_clean = appointments_clean.merge(
    doctors_clean,
    on=["doctor_id", "department"],
    how="left"
)

appointments_clean.head()

,appointment_id,slot_id,scheduling_date,appointment_date,appointment_time,scheduling_interval,status,check_in_time,appointment_duration,start_time,...,appointment_month,appointment_month_name,appointment_day_name,appointment_hour,is_attended,is_no_show,is_cancelled,waiting_time_category,doctor_name,max_appointments_per_day
0,6,1,2023-12-08,2024-01-01,08:00:00,24,attended,07:35:33,46.5,08:01:05,...,1,January,Monday,8,1,0,0,16-30 min,Doctor 10,24
1,118,21,2023-12-25,2024-01-01,13:00:00,7,did not attend,NaT,NaN,NaT,...,1,January,Monday,13,0,1,0,Not attended,Doctor 24,20
2,194,22,2023-12-29,2024-01-01,13:15:00,3,attended,13:01:40,16.9,13:28:46,...,1,January,Monday,13,1,0,0,16-30 min,Doctor 18,20
3,212,23,2023-12-30,2024-01-01,13:30:00,2,attended,13:09:37,8.3,13:46:43,...,1,January,Monday,13,1,0,0,31-45 min,Doctor 16,28
4,168,24,2023-12-28,2024-01-01,13:45:00,4,attended,13:16:48,21.8,13:55:56,...,1,January,Monday,13,1,0,0,31-45 min,Doctor 5,16


In [6]:
missing_summary = appointments_clean.isnull().sum().sort_values(ascending=False)
missing_summary[missing_summary > 0]

waiting_time            1655
start_time              1655
end_time                1655
check_in_time           1655
appointment_duration    1655
dtype: int64

In [7]:
appointments_clean.to_csv(PROCESSED_DIR / "cleaned_appointments.csv", index=False)

print("Cleaned file saved:")
print(PROCESSED_DIR / "cleaned_appointments.csv")

print("Shape:", appointments_clean.shape)

Cleaned file saved:
..\data\processed\cleaned_appointments.csv
Shape: (6837, 29)
